In [1]:
%matplotlib widget
import inspect
import re
def debugPrint(x):
    frame = inspect.currentframe().f_back
    s = inspect.getframeinfo(frame).code_context[0]
    r = re.search(r"\((.*)\)", s).group(1)
    print("{} [{}] = {}".format(r,type(x).__name__, x))
       
import torch
import numpy as np
import warp as wp

# Initialize Warp
wp.config.verify_autograd_array_access = False
wp.config.verbose = False
wp.init()

from sphWarpCore import radiusSearchCompactHashMap, sphOperation_warp
from sphWarpCore.enumTypes import *

import matplotlib.pyplot as plt
from demo_util import *
from warpPlot import *

Warp CUDA warning: Could not find or load the NVIDIA CUDA driver. Proceeding in CPU-only mode.


Warp 1.11.1 initialized:
   CUDA driver not found or failed to initialize
   Devices:
     "cpu"      : "aarch64"
   Kernel cache:
     /home/frieren/.cache/warp/1.11.1


In [2]:
device = torch.device('cpu')
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
targetNumNeighbors = 50
nx = 128
dim = 2
numParticles = nx**dim

warpOnly = False
periodic = True

kernel = KernelFunctions.Wendland2
supportMode = SupportScheme.Gather

markerSize = 8 if warpOnly else 2
gridVisualization = False 
gridResolution = 128
dx = 2.0 / nx

In [3]:
particleState, domain, adjacency, neighborhood, simulationState, measurements = prepData(nx, targetNumNeighbors, dim, device, periodic, warpOnly)
apparentVolume, crkDensity, crkState = computeCRKFactors(particleState, domain, kernel, adjacency = adjacency)

Module sphWarpCore.radiusSearch.wp_compactHash 4845d5a load on device 'cpu' took 12.73 ms  (cached)
Module sphWarpCore.operations.wp_density 14892f2 load on device 'cpu' took 1.67 ms  (cached)
Module sphWarpCore.crk.crk_volume cb121df load on device 'cpu' took 18.12 ms  (cached)
Module sphWarpCore.crk.crk_moments a970da5 load on device 'cpu' took 1.58 ms  (cached)
Module sphWarpCore.crk.crk_density 9eed5d7 load on device 'cpu' took 1.62 ms  (cached)


In [4]:
f_linear = particleState.positions[:,0] * 5 + 10
f_grad_x = torch.full_like(f_linear, 5.0)
f_grad_y = torch.zeros_like(f_linear)

linear_gradient_warp = warpOperation(
    queryParticles = particleState,
    queryValues = f_linear,
    operationProperties=OperationProperties(
        kernel = kernel,
        supportMode = supportMode,
        operation = WarpOperation.Gradient,
        gradientMode = GradientScheme.Difference,
    ),
    adjacency = adjacency,
    domain = domain,
)

print("Linear Gradient (WarpSPH): ", linear_gradient_warp)

mean_error_x = torch.mean(torch.abs(linear_gradient_warp[:,0] - f_grad_x))
mean_error_y = torch.mean(torch.abs(linear_gradient_warp[:,1] - f_grad_y))

print("Mean Absolute Error in X Gradient: ", mean_error_x.item())
print("Mean Absolute Error in Y Gradient: ", mean_error_y.item())

Module sphWarpCore.operations.wp_gradient 13b43b0 load on device 'cpu' took 1.18 ms  (cached)
Linear Gradient (WarpSPH):  tensor([[-2.1077e+02,  1.3928e+01],
        [-9.8097e+01,  6.7369e+00],
        [-3.9914e+00,  9.8616e-01],
        ...,
        [-1.2654e+01,  4.3763e-02],
        [-9.4684e+01, -2.0342e+00],
        [-2.1157e+02, -7.0975e+00]])
Mean Absolute Error in X Gradient:  5.067431926727295
Mean Absolute Error in Y Gradient:  0.18661506474018097


In [5]:
print("Testing Linear Interpolation with WarpSPH...")
print("Input Function: f(x,y) = 5x + 10", 'Min: ', torch.min(f_linear), "Max: ", torch.max(f_linear))
linear_interp = warpOperation(
    queryParticles = particleState,
    queryValues = f_linear,
    operationProperties=OperationProperties(
        kernel = kernel,
        supportMode = supportMode,
        operation = WarpOperation.Interpolate,
        gradientMode = GradientScheme.Difference,
    ),
    adjacency = None,
    domain = domain,
)
print("Linear Interpolation (WarpSPH): ", linear_interp, "Min: ", torch.min(linear_interp), "Max: ", torch.max(linear_interp))

Testing Linear Interpolation with WarpSPH...
Input Function: f(x,y) = 5x + 10 Min:  tensor(5.0162) Max:  tensor(14.9785)
Module sphWarpCore.operations_grid.wp_interpolate_grid 8bc2e25 load on device 'cpu' took 1.30 ms  (cached)
Linear Interpolation (WarpSPH):  tensor([ 8.3303,  5.9334,  5.1325,  ..., 14.8497, 14.0765, 11.8625]) Min:  tensor(5.0462) Max:  tensor(15.2339)


In [6]:
f = torch.randn(numParticles, device=device, dtype=torch.float32)

f_smoothed = f.clone()

for _ in range(4):
    f_smoothed = warpOperation(
        queryParticles = particleState,
        queryValues = f_smoothed,
        operationProperties = OperationProperties(
            kernel = kernel,
            supportMode = supportMode,
            operation = WarpOperation.Interpolate,
        ),
        adjacency = None,
        domain = domain,
    )

gradient_warp = warpOperation(
    queryParticles = particleState,
    queryValues = f_smoothed,
    operationProperties = OperationProperties(
        kernel = kernel,
        supportMode = supportMode,
        operation = WarpOperation.Gradient,
        gradientMode = GradientScheme.Difference,
    ),
    adjacency = adjacency,
    domain = domain,
)


if not warpOnly:
    gradient_diffSPH = SPHOperation(
        simulationState,
        quantity = f_smoothed,
        kernel = KernelType.Wendland2,
        neighborhood = neighborhood[0],
        kernelValues = neighborhood[1],
        operation=Operation.Gradient,
        gradientMode=GradientMode.Difference,
        supportScheme = SupportScheme.Gather,
        correctionTerms= [],
        positiveDivergence=False
    )



In [7]:
from warpPlot import *

In [ ]:
plotter = visualize(
    particleState = particleState,
    domain = domain,
    quantities = {
        "A": f_smoothed,
        "B": -f_smoothed,
    },
    plotOptions = {
        "A": PlottingOptions(
            colorMap = ColorMap.rocket,
            markerSize = 0.1,
            midPoint = 'median',
            plotTitle = "Visualization of Smoothed Quantity via Grid",
            gridVisualization = GridVisualization(
                resolution = 64,
            ),
        ),
        "B": PlottingOptions(
            colorMap = ColorMap.viridis,
            markerSize = 0.1,
            midPoint = 'median',
            plotTitle = "Visualization of Negative Smoothed Quantity via Grid",
            gridVisualization = GridVisualization(
                resolution = 64,
            ),
        ),
    },
    figTitle = "Initial Visualization of Smoothed Quantities",
    mosaic = 'AB',
    figsize= (11,5)
)

NotImplementedError: The pyvista backend is not yet implemented (Phase 3).  Track progress in TASKS_BACKEND_PLAN.md.

In [9]:
plotter.updateQuantities(
    {
        "A": f_smoothed,
        "B": -f_smoothed,
    },
    newOptions = {
        'A': {
            'colorMap': ColorMap.Spectral
        }
    },
)
    